# 3. Tương quan và hồi quy tuyến tính

Notebook cốt lõi cho Chương 4.6. Mục tiêu là minh họa mô hình tuyến tính, diễn giải hệ số,
độ phù hợp và kiểm tra giả định cơ bản. Cross-validation, HC3 joint tests và sensitivity
analysis chi tiết nằm trong `notebooks/04_regression.ipynb`.

In [ ]:
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
ALPHA = 0.05
ROOT = Path("../..")
DATA_RAW = ROOT / "data" / "raw"
DATA_OUT = ROOT / "data" / "processed"
FIGURES = ROOT / "report" / "figures"
DATA_OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error

df = pd.read_csv(DATA_OUT / "student_mat_clean.csv")

## 3.1. Tương quan giữa các điểm số

In [ ]:
corr = df[["G1", "G2", "G3", "failures", "studytime", "absences"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Tương quan Spearman giữa các biến chính")
fig.tight_layout()
fig.savefig(FIGURES / "reg_course_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
corr

## 3.2. Hai mô hình hồi quy

In [ ]:
formula_a = "G3 ~ failures + studytime + absences + C(school) + C(sex)"
formula_b = formula_a + " + G1 + G2"
model_a = smf.ols(formula_a, data=df).fit()
model_b = smf.ols(formula_b, data=df).fit()

comparison = pd.DataFrame([
    {"model": "A: không G1/G2", "r_squared": model_a.rsquared, "adjusted_r_squared": model_a.rsquared_adj,
     "rmse_in_sample": mean_squared_error(df["G3"], model_a.fittedvalues) ** 0.5},
    {"model": "B: có G1/G2", "r_squared": model_b.rsquared, "adjusted_r_squared": model_b.rsquared_adj,
     "rmse_in_sample": mean_squared_error(df["G3"], model_b.fittedvalues) ** 0.5},
])
comparison

Model A mô tả mối liên hệ với các thông tin nền và hành vi. Model B thêm `G1`, `G2`, là
điểm số rất gần thời điểm `G3`; do đó Model B phù hợp hơn cho dự báo muộn nhưng không trả
lời tốt câu hỏi can thiệp sớm.

## 3.3. Hệ số và khoảng tin cậy

In [ ]:
def coefficient_table(model, name):
    ci = model.conf_int()
    return pd.DataFrame({
        "model": name,
        "term": model.params.index,
        "coefficient": model.params.values,
        "ci_lower": ci[0].values,
        "ci_upper": ci[1].values,
        "p_value": model.pvalues.values,
    })

coefficients = pd.concat([
    coefficient_table(model_a, "A"),
    coefficient_table(model_b, "B"),
], ignore_index=True)
coefficients.to_csv(DATA_OUT / "course_regression_summary.csv", index=False)
coefficients

## 3.4. Kiểm tra residual cơ bản

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for row, (name, model) in enumerate([("Model A", model_a), ("Model B", model_b)]):
    axes[row, 0].scatter(model.fittedvalues, model.resid, alpha=0.45, s=20)
    axes[row, 0].axhline(0, color="black", linestyle="--")
    axes[row, 0].set(title=f"{name}: residual vs fitted", xlabel="Fitted G3", ylabel="Residual")
    stats.probplot(model.resid, dist="norm", plot=axes[row, 1])
    axes[row, 1].set_title(f"{name}: Q-Q plot")
fig.tight_layout()
fig.savefig(FIGURES / "reg_course_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

## Kết luận

- `G1` và đặc biệt `G2` giải thích phần lớn biến thiên của `G3` trong mô hình tuyến tính.
- Model không có điểm quá trình có khả năng giải thích hạn chế hơn.
- Residual không hoàn toàn chuẩn và nhóm `G3=0` tạo cấu trúc khó mô tả bằng OLS.
- Hệ số là association có điều kiện trên các biến đã đưa vào mô hình, không phải tác động
  nhân quả.